# Predicción del fútbol uruguayo con un árbol de decisión ID3 propio

Mediante este notebook se detalla el proceso de entrenamiento de un arbol de decision **ID3** propio (implementado en `src/id3.py`) para predecir el resultado de un partido del campeonato de primera división del futbol uruguayo (`L` local, `E` empate, `V` visitante).

Vamos a ver **que ocurre en cada paso**:

1. como se limpia el dataset original;
2. como se construyen atributos causales sin filtrar informacion del futuro;
3. como se discretizan las tasas para que ID3 pueda operar;
4. la entropia y la ganancia de informacion, es decir, como decide el arbol;
5. la evaluacion final y un analisis de la efectividad del algoritmo.

## 1. Configuración

Se importan los modulos propios de `src/` (enfocados a la tarea) y las utilidades de metricas de scikit-learn.

In [ ]:
# Configuración inicial y utilidades

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import TimeSeriesSplit

# sklearn.metrics se usa solo para evaluar el árbol una vez entrenado

# Se agrega al path la carpeta src/ para poder importar los módulos propios del proyecto
# y no instalarlos como paquete

ROOT = Path.cwd()
if not (ROOT / 'requirements.txt').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from features import (  # noqa: E402
    NUMERIC_FEATURES,
    build_causal_match_features,
    load_clean_matches,
)
from baseline import TenYearWinRateClassifier  # noqa: E402
from id3 import ID3  # noqa: E402
from preprocessing import MixedTypeDiscretizer  # noqa: E402
from evaluation import temporal_holdout, new_discretizer  # noqa: E402

RAW = ROOT / 'data/raw/futbol_uruguayo.zip'
CLASSES = ['E', 'L', 'V']

# discretización
BIN_ORDER = ['baja', 'media', 'alta']
LAST_5_CUTS = [0.3, 0.6]

RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)


def entropia(series: pd.Series) -> float:
    """Entropia de Shannon en bits de una serie de etiquetas."""
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())

## 2. Datos: lectura y limpieza del ZIP

El dataset original viene comprimido (data/raw/futbol_uruguayo.zip) con un unico CSV. load_clean_matches lo extrae en memoria, valida las columnas, quita duplicados y deriva la clase objetivo winner exclusivamente de los goles (gh/ga):

- `L` si `gh > ga`;
- `V` si `gh < ga`;
- `E` si `gh == ga`.

Los goles nunca seran atributos del modelo, solo construyen la etiqueta y los historiales de partidos anteriores.

In [ ]:
matches = load_clean_matches(RAW)
print('Filas limpias:', len(matches))
display(matches.head(3))
display(matches.dtypes.to_frame(name='tipo'))
print('Distribucion de la clase (todo el historico):')
display(
    matches['winner']
    .value_counts()
    .reindex(CLASSES)
    .rename('cantidad')
    .to_frame()
)

## 3. Atributos causales

`build_causal_match_features` construye, para cada partido, tasas historicas calculadas con fechas estrictamente anteriores. Los partidos de un mismo dia se "featurizan" todos antes de actualizar los historiales, de modo que un partido jamas usa el resultado de otro partido del mismo dia (esto evita leakage).

El modelo usa 6 tasas de victoria:

- `home/away_win_rate_last_5`: forma de cada equipo en sus ultimos 5 partidos;
- `home/away_win_rate_season`: exito de cada equipo dentro del ano calendario;
- `home_win_rate_as_home_all`: historico del local actuando de local;
- `home_win_rate_h2h_as_home`: historico del local contra ese visitante con la misma localia (head-to-head orientado).

Cuando un equipo no tiene historial (debut, primera vez en la ventana o primer partido del año por ejemplo) el denominador es 0 y se imputa un neutro de 0.5 en lugar de 0.0, esto para no confundir "nunca jugo" con "siempre perdio".

In [ ]:
featured = build_causal_match_features(matches)
print('Partidos con atributos:', len(featured))
display(featured[['date', 'home', 'away'] + NUMERIC_FEATURES + ['winner']].head(5))

sin_historial = featured[featured['home_win_rate_season'].eq(0.5)]
print('Tasa 0.5: puede ser imputada o una proporción real, no prueba ausencia de historial:')
display(sin_historial[['date', 'home', 'away'] + NUMERIC_FEATURES].head(3))

## 4. Partición temporal

El protocolo es temporal y nunca aleatorio:

- Train: partidos hasta 2023 inclusive.
- Test: partidos de 2024 y 2025.

El test queda fuera de toda decision de atributos o parametros.

In [ ]:
train, test = temporal_holdout(featured)
assert train['date'].dt.year.le(2023).all()
assert test['date'].dt.year.isin([2024, 2025]).all()
print('Train:', len(train), 'partidos (hasta 2023)')
print('Test: ', len(test), 'partidos (2024-2025)')

## 5. Discretización: que exactamente ve el arbol

ID3 es categórico: necesita valores enteros, no tasas continuas. Cada tasa se convierte en una de 3 categorias (`baja`, `media`, `alta`), mapeadas a codigos enteros `1`, `2`, `3` (el `0` queda reservado para valores desconocidos).

Dos estrategias de cortes:

- `win_rate_last_5`: cortes fijos `[0.3, 0.6]`. Como la tasa solo puede ser 0, 0.2, 0.4, ... estas categorias significan literalmente 0-1 triunfos (baja), 2 triunfos (media), 3+ triunfos (alta) de los ultimos 5. No dependen de los datos, asi que no requieren ajuste con train (y tampoco hay riesgo de leakage).
- El resto: cuantiles (equal-frequency) calculados solo con train; cada tercil cae exactamente en un tercio de las filas de entrenamiento.

Mostramos los bordes resultantes y una decodificacion visual: tasa cruda a la izquierda, codigo a la derecha.

In [ ]:
discretizer = new_discretizer()
discretizer.fit(train[NUMERIC_FEATURES])

print('Bordes de cada atributo (dos cortes -> tres bines):')
display(
    pd.DataFrame(
        {column: discretizer.numeric_edges_[column] for column in NUMERIC_FEATURES}
    ).T.rename(columns={0: 'borde_1', 1: 'borde_2'})
)

X_train = discretizer.transform(train[NUMERIC_FEATURES])
X_test = discretizer.transform(test[NUMERIC_FEATURES])

print('Ejemplo: la tasa cruda y el codigo que recibe ID3 (1=baja, 2=media, 3=alta):')
muestra = train[['home', 'away'] + NUMERIC_FEATURES].head(5).copy()
all_columns = [column for column in NUMERIC_FEATURES]
codigos = pd.DataFrame(
    X_train[:5], columns=[column + '_cod' for column in NUMERIC_FEATURES]
)
display(pd.concat([muestra, codigos], axis=1))

print('Cuantas filas de train cae en cada categoria:')
resumen = pd.DataFrame({column: pd.Series(X_train[:, i]).map(
    {1: 'baja', 2: 'media', 3: 'alta'}
).value_counts() for i, column in enumerate(NUMERIC_FEATURES)})
display(resumen.reindex(BIN_ORDER).fillna(0).astype(int))

## 6. Como decide el arbol: entropia y ganancia

ID3 elige el atributo que mas reduce la entropia (desorden) de la clase. La entropia de la clase en train es:

```text
H(Y) = -sum_c P(c) * log2(P(c))
```

que mide cuantos bits de informacion hacen falta para decir el resultado sabiendo solo la distribucion global. La ganancia de informacion de un atributo es la entropia menos la entropia condicional promedio tras partir por ese atributo.

En este paso calculamos la primera decision que tomaria el arbol mostrando la ganancia de cada atributo sobre todo train.

In [ ]:
parent_entropy = entropia(train['winner'])
print(f'Entropia de la clase en train: {parent_entropy:.4f} bits '
      '(maximo teorico ~1.585 bits para 3 clases equiprobables)')

gains = []
for index, column in enumerate(NUMERIC_FEATURES):
    conditional = 0.0
    for code in np.unique(X_train[:, index]):
        mask = X_train[:, index] == code
        conditional += mask.mean() * entropia(train['winner'][mask])
    gains.append({
        'atributo': column,
        'entropia_condicional': round(conditional, 4),
        'ganancia': round(parent_entropy - conditional, 4),
    })
gains_frame = pd.DataFrame(gains).sort_values('ganancia', ascending=False)
display(gains_frame.reset_index(drop=True))

mejor = gains_frame.iloc[0]
print(
    f'Primera division sobre {mejor["atributo"]} '
    f'(ganancia {mejor["ganancia"]:.4f} bits).'
)

## 7. Entrenamiento del árbol

El procedimiento es recursivo y greedy:

1. se calcula la ganancia de todos los atributos disponibles en el nodo;
2. se parte por el de mayor ganancia (una rama por categoria);
3. cada atributo se usa a lo sumo una vez por rama;
4. se detiene cuando un nodo es puro, no quedan atributos, o la mejor ganancia no supera `min_info_gain`.

In [ ]:
# min_info_gain=0.0 (sin poda): es la opcion con mejor macro-F1 promedio
# en la validacion cruzada temporal de la seccion 12.
id3_gain_elegido = 0.0

arbol_id3 = ID3(min_info_gain=id3_gain_elegido)
arbol_id3.fit(X_train, train['winner'])

print('Profundidad maxima:', arbol_id3.get_depth())
print('Cantidad de hojas:', arbol_id3.get_n_leaves())
print('\nImportancia de atributos (ganancia acumulada normalizada):')
importancias = pd.DataFrame({
    'atributo': NUMERIC_FEATURES,
    'importancia': arbol_id3.feature_importances_,
}).sort_values('importancia', ascending=False)
display(importancias)

## 8. Evaluacion sobre test (2024-2025)

Notar que test no participo en nada: ni en la discretizacion, ni en el arbol. Medimos:

- **accuracy**: proporcion de predicciones correctas;
- **macro-F1**: media aritmetica del F1 de cada clase (no favorece a la clase mayoritaria);
- **reporte por clase** (precision, recall, F1) coincidente con la matriz de confusion.

In [ ]:
y_test = test['winner'].to_numpy()
y_pred = arbol_id3.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Macro-F1:', round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4))
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

matrix = confusion_matrix(y_test, y_pred, labels=CLASSES)
fig, axis = plt.subplots(figsize=(5, 4))
image = axis.imshow(matrix, cmap='Blues')
axis.set_xticks(range(3), CLASSES)
axis.set_yticks(range(3), CLASSES)
axis.set_xlabel('prediccion')
axis.set_ylabel('real')
for row in range(3):
    for column in range(3):
        axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
plt.show()

## 9. Ejemplos concretos: donde acierta y donde falla

Vemos partidos reales de test con la prediccion y el resultado. La columna acierta indica si ID3 acerto.

In [ ]:
predicciones = test[['date', 'home', 'away', 'winner']].copy()
predicciones['prediccion_id3'] = y_pred
predicciones['acierta'] = predicciones['prediccion_id3'] == predicciones['winner']
display(predicciones.sample(8, random_state=RANDOM_STATE))
print('Aciertos totales:', int(predicciones['acierta'].sum()), '/', len(predicciones))

empates_reales = predicciones[predicciones['winner'] == 'E']
print('Sobre los', len(empates_reales), 'empates reales de test, '
      f'ID3 predijo exactamente empate en {(empates_reales["prediccion_id3"] == "E").sum()} '
      f'({(empates_reales["prediccion_id3"] == "E").mean():.1%}).')
display(empates_reales.head(3))

## 10. Efecto de podar con min_info_gain

Un arbol que crece demasiado memoriza ruido del train (sobreajuste). La opcion "podada" detiene la expansion cuando la ganancia no supera un umbral. Comparamos la eficacia de ambas versiones.

In [ ]:
configuraciones = [
    ('ID3 completa (gain=0.0)', ID3(min_info_gain=0.0)),
    ('ID3 podada (gain=0.02)', ID3(min_info_gain=0.02)),
]

comparativa = []

for nombre, modelo in configuraciones:
    modelo.fit(X_train, train['winner'])
    predicha = modelo.predict(X_test)
    comparativa.append({
        'modelo': nombre,
        'profundidad': modelo.get_depth(),
        'hojas': modelo.get_n_leaves(),
        'accuracy': round(accuracy_score(y_test, predicha), 4),
        'macro_f1': round(f1_score(y_test, predicha, average='macro', zero_division=0), 4),
    })
display(pd.DataFrame(comparativa))

## 11. Comparadores con parámetros ya seleccionados

Nuestro ID3 es categórico y multiway, es decir que exige discretizar y parte en tantas ramas como categorias. Compararlo con la caja de herramientas de scikit-learn permite ver cuanto de la diferencia es del algoritmo y cuanto de la representacion de los datos. Tenemos que:

- sklearn DecisionTree (mismo input): un arbol de sklearn entrenado con el mismo input discretizado que recibe nuestro ID3 (criterion='entropy'). Aisla la diferencia entre implementaciones.
- sklearn DecisionTree (tasas crudas): el mismo arbol pero sobre las tasas continuas. Puede hacer cortes binarios en cualquier umbral, sin perder la precision de la tasa original como hace la discretizacion en 3 bines.
- sklearn RandomForest (tasas crudas): bosque de 300 arboles con class_weight='balanced_subsample', que re-muestrea cada arbol para equilibrar las clases. Representante de metodos modernos para comparar.

Nota: RandomForest agrega 300 arboles; su profundidad y cantidad de hojas son por-arbol y por eso quedan vacias en la tabla.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Sin una busqueda de hiperparametros propia para el Random Forest en este
# notebook, se usan los valores por defecto de scikit-learn ademas de los
# ya fijados explicitamente (n_estimators, class_weight).
rf_params_elegidos = {}

X_train_raw = train[NUMERIC_FEATURES].to_numpy()
X_test_raw = test[NUMERIC_FEATURES].to_numpy()
y_train = train['winner'].to_numpy()
competidores = {
    'sklearn DT (mismo input)': DecisionTreeClassifier(criterion='entropy', random_state=42),
    'sklearn DT (tasas crudas)': DecisionTreeClassifier(criterion='entropy', random_state=42),
    'sklearn RandomForest (crudo)': RandomForestClassifier(
        n_estimators=300, class_weight='balanced_subsample', random_state=42,
        **rf_params_elegidos),
}

filas = []
for nombre, modelo in competidores.items():
    usa_codigos = 'mismo input' in nombre
    X_entrena = X_train if usa_codigos else X_train_raw
    X_prueba = X_test if usa_codigos else X_test_raw
    modelo.fit(X_entrena, y_train)
    predicha = modelo.predict(X_prueba)
    filas.append(
        {
            'modelo': nombre,
            'input': '3 categorias' if usa_codigos else 'tasas crudas',
            'profundidad': (modelo.get_depth()
            if hasattr(modelo, 'get_depth') else None),
            'hojas': (modelo.get_n_leaves()
          if hasattr(modelo, 'get_n_leaves') else None),
            'accuracy': round(accuracy_score(y_test, predicha), 4),
            'macro_f1': round(
                f1_score(y_test, predicha, average='macro', zero_division=0), 4
            ),
        }
    )

# Clasificador base de la letra: gana el equipo con mayor proporcion de
# victorias en los ultimos diez anos (una regla, no un modelo aprendido).
# Se entrena solo con train y se evalua sobre test, igual que el ID3.
base_10 = TenYearWinRateClassifier()
base_10.fit(train[['date', 'home', 'away']], train['winner'])
pred_base = base_10.predict(test[['date', 'home', 'away']])
filas.append({
    'modelo': 'Base 10 anios (clasificador base)',
    'input': 'fechas/equipos',
    'profundidad': None,
    'hojas': None,
    'accuracy': round(accuracy_score(y_test, pred_base), 4),
    'macro_f1': round(
        f1_score(y_test, pred_base, average='macro', zero_division=0), 4
    ),
})

display(pd.DataFrame(filas))

## 12. Validación cruzada temporal (ID3 propio)

La evaluación final usa un solo split (train <= 2023, test 2024-2025). Para ver si los numeros dependen de ese split en particular, cruzamos con TimeSeriesSplit de scikit-learn, particiones cronológicas que solo usan pasado para predecir futuro (sin barajar). El discretizador se "re-entrena" dentro de cada fold, de modo que sus estadisticas usan solo el train de ese fold (igual que en la corrida principal). Barremos min_info_gain para comparar con el valor 0.0 que usamos arriba.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)
historico = featured[featured['year'] <= 2023].sort_values('date').reset_index(drop=True)

lineas_cv = {}
for min_gain in [0.0, 0.01, 0.02]:
    accs, f1s = [], []
    for train_idx, val_idx in folds.split(historico):
        tr = historico.iloc[train_idx]
        va = historico.iloc[val_idx]
        disc_cv = MixedTypeDiscretizer(
            categorical_features=[],
            numeric_features=NUMERIC_FEATURES,
            n_bins=3,
            fixed_cuts={'home_win_rate_last_5': LAST_5_CUTS,
                        'away_win_rate_last_5': LAST_5_CUTS},
        )
        disc_cv.fit(tr[NUMERIC_FEATURES])
        arbol_cv = ID3(min_info_gain=min_gain)
        arbol_cv.fit(disc_cv.transform(tr[NUMERIC_FEATURES]),
                     tr['winner'].to_numpy())
        pred_cv = arbol_cv.predict(disc_cv.transform(va[NUMERIC_FEATURES]))
        accs.append(accuracy_score(va['winner'], pred_cv))
        f1s.append(f1_score(va['winner'], pred_cv,
                           average='macro', zero_division=0))
    print(f'min_info_gain={min_gain:5}: acc={np.mean(accs):.4f} '
          f'(+/- {np.std(accs):.4f}) | macro-F1={np.mean(f1s):.4f} '
          f'(+/- {np.std(f1s):.4f})')
    lineas_cv[min_gain] = (np.mean(accs), np.mean(f1s))

print('\nMejor min_info_gain por accuracy promedio en CV:')
print(' ', max(lineas_cv, key=lambda k: lineas_cv[k][0]))
print('Mejor min_info_gain por macro-F1 promedio en CV:')
print(' ', max(lineas_cv, key=lambda k: lineas_cv[k][1]))

## 13. Efectividad del algoritmo

Para interpretar los numeros hay que compararlos con los baselines: predecir siempre L (la clase mayoritaria de test) y la base de 10 años (el clasificador oficial de la letra, seccion 11). Tambien revisamos la distribución de predicciones y el recall por clase.

In [ ]:
baseline_trivial = float((test['winner'] == 'L').mean())
print('Accuracy de predecir siempre L:', round(baseline_trivial, 4))
print('Accuracy de la base 10 anios:   ', round(accuracy_score(y_test, pred_base), 4))
print(f'Accuracy del ID3 (gain={id3_gain_elegido:g}):', round(accuracy_score(y_test, y_pred), 4))
print('\nDistribucion de predicciones del ID3 en test:')
display(
    pd.Series(y_pred)
    .value_counts()
    .reindex(CLASSES)
    .rename('predicciones')
    .to_frame()
)

reporte = classification_report(
    y_test, y_pred, output_dict=True, zero_division=0
)
print('Precision/Recall/F1 por clase:')
for clase in CLASSES:
    valor = reporte[clase]
    print(f'  {clase}: precision={valor["precision"]:.3f} '
          f'recall={valor["recall"]:.3f} f1={valor["f1-score"]:.3f}')

### Cómo interpretar estos resultados

Esta sección responde dos preguntas centrales. ¿El modelo funciona? y ¿por qué no rinde más?

1. El modelo supera al azar y a los baselines. Con tres clases, un clasificador aleatorio rondaría el 33% de accuracy; predecir siempre L alcanza aproximadamente 40%, dado que la victoria local es la clase mayoritaria; y la regla de los diez años (el clasificador base de la cátedra) obtiene su propio nivel de accuracy. En las tablas de las secciones 11 y 12, el ID3 supera a ambas reglas simples y, a diferencia de un baseline fijo, es capaz de capturar victorias visitantes. Cabe señalar, con honestidad, que la ventaja sobre la regla de los diez años es pequeña (alrededor de 0.4 puntos porcentuales), lo que sugiere que las tasas discretizadas ya están cerca de su techo informativo. La validación cruzada temporal (sección 12) confirma esta lectura: un valor moderado de min_info_gain mantiene o incrementa levemente el accuracy promedio entre folds históricos, pero reduce el macro-F1, a costa de las clases minoritarias.

2. La clase E (empate) resulta casi imposible de separar. Los empates representan entre 27% y 28% de los datos, y su recall se ubica entre 17% y 18%. Existe una razón estructural para esto: los atributos utilizados son tasas de victoria que, por su propia construcción, no discriminan empates; un equipo en excelente forma puede igualmente empatar en la práctica. En consecuencia, el algoritmo concentra la ganancia de información en separar L de V, mientras que los empates quedan diluidos como ruido dentro de cada hoja, que tiende a predecir la opción de menor riesgo y rara vez asigna la clase E.

3. La naturaleza voraz del algoritmo y la discretización en categorías comprimen información. ID3 selecciona los atributos de manera voraz, evaluándolos uno por uno sin considerar combinaciones entre ellos, y la discretización en tres bins reduce la precisión de la tasa original: por ejemplo, distingue 2 de 5 de 3 de 5, pero no diferencia 0.40 de 0.45. Dado que este problema presenta un ruido considerable en cada partido, el árbol alcanza un macro-F1 de aproximadamente 0.41, un valor razonable que, sin embargo, evidencia el límite impuesto por la naturaleza de los atributos disponibles.

4. La poda no siempre mejora el resultado. La poda, con un umbral de ganancia de 0.02, simplifica considerablemente el árbol, de aproximadamente 645 hojas a solo 7, y en ocasiones incrementa el accuracy, pero a costa de reducir el macro-F1, ya que deja de predecir las clases minoritarias, es decir mejora el desempeño en los casos más sencillos, pero pierde equilibrio entre clases. El hiperparámetro min_info_gain permite controlar este compromiso, aunque no existe una configuración óptima universal.